Actividad numero 1 

In [17]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config_conexion import obtener_motor
from esquema import TABLAS, COLUMNAS
from sqlalchemy import text

motor = obtener_motor()


Comprobar la conexión real con SQL Server

In [14]:
with motor.connect() as conexion:
    resultado = conexion.execute(text("SELECT 1")).scalar()

print("Conexión SQL Server correcta:", resultado)

Conexión SQL Server correcta: 1


In [15]:
import pandas as pd
import time

# Fecha utilizada como parámetro de la consulta
fecha_corte = '2005-01-01'


# ==========================================================
# 1. CONSTRUCCIÓN DEL CONJUNTO DE TRABAJO
# ==========================================================

# Los nombres de tablas y columnas se obtienen exclusivamente
# desde esquema.py.
#
# Los valores variables, como fecha_corte, se envían mediante
# params para evitar concatenarlos directamente en la consulta.

sql_raw = f"""
SELECT 
    a.{COLUMNAS['rental_id']}        AS rental_id,
    a.{COLUMNAS['rental_date']}      AS rental_date,
    a.{COLUMNAS['return_date']}      AS return_date,

    c.{COLUMNAS['customer_id']}      AS customer_id,
    c.{COLUMNAS['first_name']}       AS first_name,
    c.{COLUMNAS['last_name']}        AS last_name,

    p.{COLUMNAS['film_id']}          AS film_id,
    p.{COLUMNAS['title']}            AS title,
    p.{COLUMNAS['rental_duration']}  AS rental_duration,
    p.{COLUMNAS['replacement_cost']} AS replacement_cost,
    p.{COLUMNAS['rental_rate']}      AS rental_rate,

    cat.{COLUMNAS['name']}           AS category

FROM {TABLAS['rental']} a

INNER JOIN {TABLAS['inventory']} i 
    ON a.{COLUMNAS['inventory_id']} = i.{COLUMNAS['inventory_id']}

INNER JOIN {TABLAS['film']} p 
    ON i.{COLUMNAS['film_id']} = p.{COLUMNAS['film_id']}

INNER JOIN {TABLAS['customer']} c 
    ON a.{COLUMNAS['customer_id']} = c.{COLUMNAS['customer_id']}

LEFT JOIN {TABLAS['film_category']} pc 
    ON p.{COLUMNAS['film_id']} = pc.{COLUMNAS['film_id']}

LEFT JOIN {TABLAS['category']} cat 
    ON pc.{COLUMNAS['category_id']} = cat.{COLUMNAS['category_id']}

WHERE a.{COLUMNAS['rental_date']} >= :fecha

ORDER BY a.{COLUMNAS['rental_date']} ASC
"""

query_extraccion = text(sql_raw)

print("Cargando conjunto de trabajo...")

df_alquileres = pd.read_sql(
    query_extraccion,
    motor,
    params={"fecha": fecha_corte}
)

# Conversión explícita de fechas
df_alquileres['rental_date'] = pd.to_datetime(
    df_alquileres['rental_date']
)

df_alquileres['return_date'] = pd.to_datetime(
    df_alquileres['return_date']
)

print(f"Filas recuperadas: {len(df_alquileres)}")
print(f"Columnas recuperadas: {len(df_alquileres.columns)}")

print("\nPrimeras filas:")
print(df_alquileres.head())


# ==========================================================
# 2. COMPARACIÓN DE UNA MISMA MÉTRICA EN SQL Y PANDAS
# ==========================================================
#
# Métrica elegida:
# promedio de días de atraso de los alquileres DEVUELTOS.
#
# Un alquiler devuelto antes de la fecha pactada tiene
# 0 días de atraso y no un valor negativo.
#
# Los alquileres sin devolución se excluyen de esta métrica
# porque su atraso definitivo todavía no se conoce.


# ----------------------------------------------------------
# ENFOQUE A - MÉTRICA CALCULADA EN SQL SERVER
# ----------------------------------------------------------

query_metrica_sql = text(f"""
SELECT
    AVG(
        CAST(
            CASE
                WHEN DATEDIFF(
                    SECOND,
                    a.{COLUMNAS['rental_date']},
                    a.{COLUMNAS['return_date']}
                ) / 86400.0 - p.{COLUMNAS['rental_duration']} > 0

                THEN DATEDIFF(
                    SECOND,
                    a.{COLUMNAS['rental_date']},
                    a.{COLUMNAS['return_date']}
                ) / 86400.0 - p.{COLUMNAS['rental_duration']}

                ELSE 0
            END
        AS FLOAT)
    ) AS promedio_dias_atraso

FROM {TABLAS['rental']} a

INNER JOIN {TABLAS['inventory']} i
    ON a.{COLUMNAS['inventory_id']} = i.{COLUMNAS['inventory_id']}

INNER JOIN {TABLAS['film']} p
    ON i.{COLUMNAS['film_id']} = p.{COLUMNAS['film_id']}

WHERE
    a.{COLUMNAS['rental_date']} >= :fecha
    AND a.{COLUMNAS['return_date']} IS NOT NULL
""")


t0_sql = time.perf_counter()

df_res_sql = pd.read_sql(
    query_metrica_sql,
    motor,
    params={"fecha": fecha_corte}
)

tiempo_sql = time.perf_counter() - t0_sql

promedio_sql = float(
    df_res_sql.iloc[0]['promedio_dias_atraso']
)


# ----------------------------------------------------------
# ENFOQUE B - MISMA MÉTRICA CALCULADA EN PANDAS
# ----------------------------------------------------------

t0_pd = time.perf_counter()

df_devueltos = df_alquileres[
    df_alquileres['return_date'].notna()
].copy()

df_devueltos['dias_prestamo'] = (
    df_devueltos['return_date']
    - df_devueltos['rental_date']
).dt.total_seconds() / 86400.0

df_devueltos['dias_atraso'] = (
    df_devueltos['dias_prestamo']
    - df_devueltos['rental_duration']
).clip(lower=0)

promedio_pandas = df_devueltos['dias_atraso'].mean()

tiempo_pandas = time.perf_counter() - t0_pd


# ==========================================================
# 3. RESULTADOS
# ==========================================================

diferencia = abs(
    promedio_sql - promedio_pandas
)

print("\n==================================================")
print("COMPARACIÓN SQL SERVER VS PANDAS")
print("==================================================")

print(
    f"Promedio atraso SQL Server : "
    f"{promedio_sql:.6f} días"
)

print(
    f"Promedio atraso Pandas     : "
    f"{promedio_pandas:.6f} días"
)

print(
    f"Diferencia                 : "
    f"{diferencia:.10f}"
)

print(
    f"Tiempo SQL Server          : "
    f"{tiempo_sql:.6f} segundos"
)

print(
    f"Tiempo Pandas              : "
    f"{tiempo_pandas:.6f} segundos"
)

if diferencia < 0.000001:
    print("\nVALIDACIÓN: ambos métodos producen el mismo resultado.")
else:
    print("\nATENCIÓN: existe una diferencia entre ambos cálculos.")


# ==========================================================
# 4. CONCLUSIÓN DEL REPARTO DE TRABAJO
# ==========================================================

print("\n==================================================")
print("CONCLUSIÓN ACTIVIDAD 1")
print("==================================================")

print("""
SQL Server se utiliza para realizar las uniones entre las tablas,
aplicar filtros y reducir el conjunto de datos antes de transferirlo
a Python.

Pandas se utiliza posteriormente para el análisis exploratorio,
control de calidad, cálculo de métricas derivadas y visualización.

Los valores variables de las consultas se envían mediante parámetros.
Los nombres de tablas y columnas no provienen del usuario, sino del
diccionario controlado esquema.py.

La comparación de la métrica de atraso permite comprobar que el cálculo
realizado en SQL Server coincide con el realizado posteriormente en
pandas.
""")

Cargando conjunto de trabajo...
Filas recuperadas: 16044
Columnas recuperadas: 12

Primeras filas:
   rental_id         rental_date         return_date  customer_id first_name  \
0          1 2005-05-24 22:53:30 2005-05-26 22:04:30          130  CHARLOTTE   
1          2 2005-05-24 22:54:33 2005-05-28 19:40:33          459      TOMMY   
2          3 2005-05-24 23:03:39 2005-06-01 22:12:39          408     MANUEL   
3          4 2005-05-24 23:04:41 2005-06-03 01:43:41          333     ANDREW   
4          5 2005-05-24 23:05:21 2005-06-02 04:33:21          222    DELORES   

  last_name  film_id            title  rental_duration  replacement_cost  \
0    HUNTER       80  BLANKET BEVERLY                7             21.99   
1   COLLAZO      333     FREAKY POCUS                7             16.99   
2   MURRELL      373    GRADUATE LORD                7             14.99   
3     PURDY      535    LOVE SUICIDES                6             21.99   
4    HANSEN      450  IDOLS SNATCHERS   

Punto 2

In [18]:

import pandas as pd
from sqlalchemy import text



# ==========================================================
# ACTIVIDAD 2 - PERFILADO Y CALIDAD DE DATOS
# ==========================================================


# ==========================================================
# 1. CARGAR EL CONJUNTO DE TRABAJO
# ==========================================================

query_raw = f"""
SELECT 
    a.{COLUMNAS['rental_id']}        AS rental_id,
    a.{COLUMNAS['rental_date']}      AS rental_date,
    a.{COLUMNAS['return_date']}      AS return_date,

    c.{COLUMNAS['customer_id']}      AS customer_id,
    c.{COLUMNAS['first_name']}       AS first_name,
    c.{COLUMNAS['last_name']}        AS last_name,

    p.{COLUMNAS['film_id']}          AS film_id,
    p.{COLUMNAS['title']}            AS title,
    p.{COLUMNAS['rental_duration']}  AS rental_duration,
    p.{COLUMNAS['replacement_cost']} AS replacement_cost,
    p.{COLUMNAS['rental_rate']}      AS rental_rate,

    cat.{COLUMNAS['name']}           AS category

FROM {TABLAS['rental']} a

INNER JOIN {TABLAS['inventory']} i 
    ON a.{COLUMNAS['inventory_id']} = i.{COLUMNAS['inventory_id']}

INNER JOIN {TABLAS['film']} p 
    ON i.{COLUMNAS['film_id']} = p.{COLUMNAS['film_id']}

INNER JOIN {TABLAS['customer']} c 
    ON a.{COLUMNAS['customer_id']} = c.{COLUMNAS['customer_id']}

LEFT JOIN {TABLAS['film_category']} pc 
    ON p.{COLUMNAS['film_id']} = pc.{COLUMNAS['film_id']}

LEFT JOIN {TABLAS['category']} cat 
    ON pc.{COLUMNAS['category_id']} = cat.{COLUMNAS['category_id']}
"""

df = pd.read_sql(
    text(query_raw),
    motor
)

# Conversión explícita de fechas
df['rental_date'] = pd.to_datetime(
    df['rental_date']
)

df['return_date'] = pd.to_datetime(
    df['return_date']
)


# ==========================================================
# 2. RECONOCIMIENTO GENERAL DEL DATASET
# ==========================================================

print("\n==================================================")
print("1. RECONOCIMIENTO GENERAL")
print("==================================================")

print(
    f"\nDimensiones: "
    f"{df.shape[0]} filas x {df.shape[1]} columnas"
)

print("\nTIPOS DE DATOS:")
print(df.dtypes)

print("\nPRIMERAS FILAS:")
print(df.head())

print("\nMUESTRA ALEATORIA:")
print(
    df.sample(
        min(5, len(df)),
        random_state=42
    )
)

print("\nINFORMACIÓN GENERAL:")
df.info()

print("\nESTADÍSTICAS DESCRIPTIVAS:")

columnas_numericas = [
    'rental_duration',
    'replacement_cost',
    'rental_rate'
]

print(
    df[columnas_numericas].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# ==========================================================
# 3. VALORES NULOS
# ==========================================================

print("\n==================================================")
print("2. VALORES NULOS")
print("==================================================")

nulos = df.isnull().sum()

pct_nulos = (
    nulos / len(df)
) * 100

df_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos
})

print(df_nulos)


# ==========================================================
# 4. DUPLICADOS SEGÚN CLAVE DE NEGOCIO
# ==========================================================

print("\n==================================================")
print("3. DUPLICADOS")
print("==================================================")

duplicados_pk = df.duplicated(
    subset=['rental_id']
).sum()

print(
    f"Duplicados por rental_id: "
    f"{duplicados_pk}"
)


# ==========================================================
# 5. CONTROL ESPECÍFICO:
#    ALQUILERES SIN DEVOLUCIÓN
# ==========================================================

print("\n==================================================")
print("4. ALQUILERES SIN DEVOLUCIÓN")
print("==================================================")

sin_devolucion = df[
    df['return_date'].isna()
].copy()

total_sin_dev = len(
    sin_devolucion
)

pct_sin_dev = (
    total_sin_dev / len(df)
) * 100

print(
    f"Cantidad: {total_sin_dev}"
)

print(
    f"Porcentaje: {pct_sin_dev:.2f}%"
)

print("""
Diagnóstico:
Una fecha de devolución NULL no se interpreta automáticamente
como un error de calidad.

En esta variante representa un dato censurado a derecha:
el alquiler todavía no registra una devolución y, por lo tanto,
no se conoce su atraso definitivo.

Estos registros se conservan y se analizan separadamente.
""")


# ==========================================================
# 6. REGLA DE NEGOCIO 1
#    DEVOLUCIÓN ANTERIOR AL ALQUILER
# ==========================================================

print("\n==================================================")
print("5. REGLAS DE NEGOCIO")
print("==================================================")

viajes_tiempo = df[
    df['return_date'].notna()
    &
    (
        df['return_date']
        < df['rental_date']
    )
]

print(
    "Regla 1 - devolución anterior al alquiler: "
    f"{len(viajes_tiempo)} violaciones"
)


# ==========================================================
# 7. CONTROL ESPECÍFICO:
#    PERÍODO PACTADO DE ALQUILER
# ==========================================================

print("\nDistribución de rental_duration:")

print(
    df['rental_duration'].describe()
)

# Una duración pactada debe ser positiva.
# Como control adicional se marca como sospechoso cualquier
# valor superior a 30 días.

periodos_invalidos = df[
    (df['rental_duration'] <= 0)
    |
    (df['rental_duration'] > 30)
]

print(
    "Períodos pactados fuera del rango razonable: "
    f"{len(periodos_invalidos)}"
)


# ==========================================================
# 8. REGLA DE NEGOCIO 2
#    COSTO DE REPOSICIÓN
# ==========================================================

costos_invalidos = df[
    df['replacement_cost'] <= 0
]

print(
    "Regla 2 - costo de reposición <= 0: "
    f"{len(costos_invalidos)} violaciones"
)


# ==========================================================
# 9. CÁLCULO DE DÍAS DE PRÉSTAMO Y DÍAS DE ATRASO
# ==========================================================

df_devueltos = df[
    df['return_date'].notna()
].copy()

df_devueltos['dias_prestamo'] = (
    df_devueltos['return_date']
    -
    df_devueltos['rental_date']
).dt.total_seconds() / 86400.0


# Métrica principal:
#
# atraso = duración real - duración pactada
#
# Si el resultado es negativo significa que fue devuelto
# antes del vencimiento, por lo que el atraso se considera 0.

df_devueltos['dias_atraso'] = (
    df_devueltos['dias_prestamo']
    -
    df_devueltos['rental_duration']
).clip(lower=0)


print("\nResumen de días de atraso:")

print(
    df_devueltos['dias_atraso'].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# ==========================================================
# 10. OUTLIERS MEDIANTE IQR
# ==========================================================

print("\n==================================================")
print("6. OUTLIERS DE DÍAS DE ATRASO - MÉTODO IQR")
print("==================================================")

q1 = df_devueltos[
    'dias_atraso'
].quantile(0.25)

q3 = df_devueltos[
    'dias_atraso'
].quantile(0.75)

iqr = q3 - q1

lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

outliers = df_devueltos[
    (
        df_devueltos['dias_atraso']
        < lim_inf
    )
    |
    (
        df_devueltos['dias_atraso']
        > lim_sup
    )
]

print(
    f"Q1: {q1:.2f} días"
)

print(
    f"Q3: {q3:.2f} días"
)

print(
    f"IQR: {iqr:.2f} días"
)

print(
    f"Límite inferior: "
    f"{lim_inf:.2f} días"
)

print(
    f"Límite superior: "
    f"{lim_sup:.2f} días"
)

print(
    f"Cantidad de outliers: "
    f"{len(outliers)}"
)


# ==========================================================
# 11. INSPECCIÓN DE OUTLIERS
# ==========================================================

if len(outliers) > 0:

    print("\nEjemplos de valores atípicos:")

    columnas_outliers = [
        'rental_id',
        'customer_id',
        'title',
        'rental_date',
        'return_date',
        'rental_duration',
        'dias_prestamo',
        'dias_atraso'
    ]

    print(
        outliers[
            columnas_outliers
        ]
        .sort_values(
            'dias_atraso',
            ascending=False
        )
        .head(20)
    )

print("""
Los valores identificados mediante IQR no se eliminan
automáticamente.

El método IQR identifica observaciones estadísticamente
atípicas, pero eso no demuestra que sean errores.

Se conservan mientras no violen una regla de negocio o exista
evidencia adicional de un error de carga.
""")


# ==========================================================
# 12. CONTROL ESPEJO EN SQL SERVER
# ==========================================================

print("\n==================================================")
print("7. COMPARACIÓN CON SQL SERVER")
print("==================================================")

query_espejo_raw = f"""
SELECT

    COUNT(
        CASE
            WHEN {COLUMNAS['return_date']} IS NULL
            THEN 1
        END
    ) AS nulos_devolucion,

    COUNT(
        CASE
            WHEN {COLUMNAS['return_date']}
                 < {COLUMNAS['rental_date']}
            THEN 1
        END
    ) AS violaciones_fechas

FROM {TABLAS['rental']}
"""

df_espejo = pd.read_sql(
    text(query_espejo_raw),
    motor
)

sql_nulos = int(
    df_espejo.iloc[0][
        'nulos_devolucion'
    ]
)

sql_fechas = int(
    df_espejo.iloc[0][
        'violaciones_fechas'
    ]
)

print(
    "Alquileres sin devolución:"
)

print(
    f"SQL Server: {sql_nulos}"
)

print(
    f"Pandas:     {total_sin_dev}"
)

print(
    "\nDevoluciones anteriores al alquiler:"
)

print(
    f"SQL Server: {sql_fechas}"
)

print(
    f"Pandas:     {len(viajes_tiempo)}"
)


# ==========================================================
# 13. VALIDACIÓN AUTOMÁTICA
# ==========================================================

print("\n==================================================")
print("8. VALIDACIÓN")
print("==================================================")

if sql_nulos == total_sin_dev:
    print(
        "OK - El conteo de alquileres sin devolución coincide."
    )
else:
    print(
        "ERROR - El conteo de alquileres sin devolución no coincide."
    )

if sql_fechas == len(viajes_tiempo):
    print(
        "OK - El control de fechas coincide."
    )
else:
    print(
        "ERROR - El control de fechas no coincide."
    )


# ==========================================================
# 14. RESUMEN DEL DIAGNÓSTICO
# ==========================================================

print("\n==================================================")
print("RESUMEN DEL DIAGNÓSTICO")
print("==================================================")

print(
    f"Registros analizados: {len(df)}"
)

print(
    f"Alquileres devueltos: {len(df_devueltos)}"
)

print(
    f"Alquileres sin devolución: {total_sin_dev}"
)

print(
    f"Duplicados: {duplicados_pk}"
)

print(
    f"Violaciones temporales: {len(viajes_tiempo)}"
)

print(
    f"Períodos pactados sospechosos: "
    f"{len(periodos_invalidos)}"
)

print(
    f"Costos de reposición inválidos: "
    f"{len(costos_invalidos)}"
)

print(
    f"Outliers de atraso mediante IQR: "
    f"{len(outliers)}"
)

print("""
Decisión general de limpieza:

- No eliminar los alquileres sin devolución, porque representan
  observaciones censuradas relevantes para el análisis de morosidad.

- No imputar una fecha de devolución ficticia.

- No eliminar automáticamente los valores atípicos identificados
  mediante IQR.

- Investigar y excluir únicamente registros que violen reglas de
  negocio comprobables, documentando la decisión tomada.

- Mantener separados los alquileres devueltos y los no devueltos
  cuando se calculen métricas cuyo valor definitivo dependa de la
  fecha de devolución.
""")


1. RECONOCIMIENTO GENERAL

Dimensiones: 16044 filas x 12 columnas

TIPOS DE DATOS:
rental_id                    int64
rental_date         datetime64[us]
return_date         datetime64[us]
customer_id                  int64
first_name                     str
last_name                      str
film_id                      int64
title                          str
rental_duration              int64
replacement_cost           float64
rental_rate                float64
category                       str
dtype: object

PRIMERAS FILAS:
   rental_id         rental_date         return_date  customer_id first_name  \
0          1 2005-05-24 22:53:30 2005-05-26 22:04:30          130  CHARLOTTE   
1          2 2005-05-24 22:54:33 2005-05-28 19:40:33          459      TOMMY   
2          3 2005-05-24 23:03:39 2005-06-01 22:12:39          408     MANUEL   
3          4 2005-05-24 23:04:41 2005-06-03 01:43:41          333     ANDREW   
4          5 2005-05-24 23:05:21 2005-06-02 04:33:21          222

Punto 3

In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

os.makedirs("graficos", exist_ok=True)
# =============================================================================
# 1. EXTRACCIÓN (Desacoplada con esquema.py)
# =============================================================================
query_raw = f"""
SELECT 
    a.{COLUMNAS['rental_id']}        AS rental_id,
    a.{COLUMNAS['rental_date']}      AS rental_date,
    a.{COLUMNAS['return_date']}      AS return_date,
    c.{COLUMNAS['customer_id']}      AS customer_id,
    CONCAT(c.{COLUMNAS['first_name']}, ' ', c.{COLUMNAS['last_name']}) AS full_name,
    p.{COLUMNAS['film_id']}          AS film_id,
    p.{COLUMNAS['rental_duration']}  AS rental_duration,
    CAST(p.{COLUMNAS['replacement_cost']} AS FLOAT) AS replacement_cost,
    cat.{COLUMNAS['name']}           AS category
FROM {TABLAS['rental']} a
INNER JOIN {TABLAS['inventory']} i ON a.{COLUMNAS['inventory_id']} = i.{COLUMNAS['inventory_id']}
INNER JOIN {TABLAS['film']} p ON i.{COLUMNAS['film_id']} = p.{COLUMNAS['film_id']}
INNER JOIN {TABLAS['customer']} c ON a.{COLUMNAS['customer_id']} = c.{COLUMNAS['customer_id']}
LEFT JOIN {TABLAS['film_category']} pc ON p.{COLUMNAS['film_id']} = pc.{COLUMNAS['film_id']}
LEFT JOIN {TABLAS['category']} cat ON pc.{COLUMNAS['category_id']} = cat.{COLUMNAS['category_id']};
"""

df = pd.read_sql(text(query_raw), motor)
df['rental_date'] = pd.to_datetime(df['rental_date'])
df['return_date'] = pd.to_datetime(df['return_date'])

# =============================================================================
# 2. MÉTRICA DERIVADA
# =============================================================================
fecha_corte = max(df['rental_date'].max(), df['return_date'].max())
fecha_efectiva = df['return_date'].fillna(fecha_corte)

dias_transcurridos = (fecha_efectiva - df['rental_date']).dt.total_seconds() / 86400.0
df['dias_atraso'] = np.maximum(0.0, dias_transcurridos - df['rental_duration'])
df['esta_devuelto'] = df['return_date'].notnull()
df['es_moroso'] = df['dias_atraso'] > 0

resumen_clientes = df.groupby(['customer_id', 'full_name']).agg(
    total_alquileres=('rental_id', 'count'),
    alquileres_con_mora=('es_moroso', 'sum'),
    exposicion_no_devueltos=('replacement_cost', lambda x: x[~df.loc[x.index, 'esta_devuelto']].sum())
).reset_index()
resumen_clientes['tasa_morosidad'] = (resumen_clientes['alquileres_con_mora'] / resumen_clientes['total_alquileres']) * 100

print("Cálculos vectorizados completados. Generando gráficos...")

# =============================================================================
# 3. VISUALIZACIÓN DE DATOS 
# =============================================================================


plt.figure(figsize=(12, 6))
df_con_atraso = df[df['dias_atraso'] > 0]
orden_cat = df_con_atraso.groupby('category')['dias_atraso'].median().sort_values(ascending=False).index

sns.boxplot(data=df_con_atraso, x='category', y='dias_atraso', order=orden_cat, palette='Set2')
plt.title("Distribución de Días de Atraso por Categoría de Película (Solo Alquileres con Mora)", fontsize=13, weight='bold')
plt.xlabel("Categoría de Película", fontsize=11)
plt.ylabel("Días de Atraso Acumulados", fontsize=11)
plt.xticks(rotation=45, ha='right')


plt.ylim(-1, 15)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("graficos/grafico_1_boxplot.png", dpi=150)
plt.close()


top_morosos = resumen_clientes.sort_values(by='exposicion_no_devueltos', ascending=False).head(10).copy()
plt.figure(figsize=(10, 6))

colores = ['#d9534f' if i == 0 else '#6c757d' for i in range(len(top_morosos))]
bars = plt.barh(top_morosos['full_name'][::-1], top_morosos['exposicion_no_devueltos'][::-1], color=colores[::-1])

plt.title("Riesgo Crítico: Una sola persona concentra la mayor deuda en películas retenidas", fontsize=13, weight='bold')
plt.xlabel("Exposición por Costo de Reposición (USD)", fontsize=11)
plt.ylabel("Cliente", fontsize=11)

max_exposicion = top_morosos.iloc[0]['exposicion_no_devueltos']
max_nombre = top_morosos.iloc[0]['full_name']
plt.annotate(
    f"Exposición máxima:\n${max_exposicion:.2f} USD",
    xy=(max_exposicion, 9),
    xytext=(max_exposicion * 0.70, 7.5),
    arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.5),
    fontsize=10, weight='semibold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.8)
)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("graficos/grafico_2_narrativo.png", dpi=150)
plt.close()


plt.figure(figsize=(10, 5))
sns.histplot(resumen_clientes['tasa_morosidad'], bins=20, color='#0275d8')
plt.title("Distribución de la Tasa de Morosidad en el Padrón de Clientes", fontsize=13, weight='bold')
plt.xlabel("Porcentaje de Alquileres Devueltos Fuera de Término (%)", fontsize=11)
plt.ylabel("Cantidad de Clientes", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("graficos/grafico_3_distribucion_morosidad.png", dpi=150)
plt.close()

print("Gráficos exportados exitosamente en la carpeta 'graficos/'.")

Cálculos vectorizados completados. Generando gráficos...


C:\Users\franc\AppData\Local\Temp\ipykernel_22440\996671689.py:64: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_con_atraso, x='category', y='dias_atraso', order=orden_cat, palette='Set2')
findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Gráficos exportados exitosamente en la carpeta 'graficos/'.
